# v33 unified — Horn/proof-grounded YNU rescue

**Baseline locked:** v32.2 (`macro-F1 = 0.596858548901435`).

This notebook is deliberately conservative:

1. Load v32.2 best preds as the baseline.
2. Audit remaining non-MC YNU errors/candidates using proof-grounded textual/FOL/Horn-style signals.
3. Build a candidate v33 only from proof-like evidence, without using gold labels in the selection rule.
4. Select v33 only if saved preds reload/recompute correctly and beat v32.2 by a strict gate.
5. Otherwise rollback cleanly to v32.2.

Important: this is **not** LoRA advocate generation. It is a CPU-only proof/closure-oriented rescue attempt.

In [ ]:
# ============================================================
# Config
# ============================================================
from pathlib import Path
import json, re, math, statistics, os, sys, traceback
from collections import Counter, defaultdict

BASELINE_MACRO_EXPECTED = 0.596858548901435
BASELINE_ACC_EXPECTED = 0.6410256410256411
BANKED_INDICES = {14, 71, 109, 125}  # v32.2 locked bank; do not touch
LABELS = ['A','B','C','D','Yes','No','Unknown']
YNU_LABELS = {'Yes','No','Unknown','UNPARSEABLE'}

# Strict selection gate for ordinary Yes/No flips. v32.2 had a special UNPARSEABLE exception;
# v33 returns to strict mode.
STRICT_MACRO_GAIN = 0.010
MAX_STANDARD_FLIPS = 12
MAX_NO_F1_DROP = 0.020
MAX_UNKNOWN_F1_DROP = 0.000
ALLOW_MC_CHANGE = False

CANDIDATE_DIRS = [
    Path('/kaggle/working'),
    Path('/kaggle/input/datasets/yixuanisthebest/v2333333'),
    Path('/kaggle/input/datasets/nguyenminhtric/test-pipeline'),
    Path('/kaggle/input'),
    Path('/mnt/data'),
    Path('.'),
]
print('Candidate dirs:')
for d in CANDIDATE_DIRS:
    print(' -', d)

In [ ]:
# ============================================================
# File helpers
# ============================================================
def find_file(names, required=True):
    if isinstance(names, str):
        names = [names]
    # exact in candidate dirs
    for d in CANDIDATE_DIRS:
        for name in names:
            p = d / name
            if p.exists():
                return p
    # recursive fallback, bounded to avoid huge searches
    for d in CANDIDATE_DIRS:
        if not d.exists() or not d.is_dir():
            continue
        try:
            for name in names:
                hits = list(d.rglob(name))[:5]
                if hits:
                    return hits[0]
        except Exception:
            pass
    if required:
        raise FileNotFoundError(f'Could not find any of: {names}')
    return None

def load_json(names, required=True):
    p = find_file(names, required=required)
    if p is None:
        return None, None
    with open(p, 'r', encoding='utf-8') as f:
        return json.load(f), p

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    return path

def copy_rows(rows):
    return [dict(r) for r in rows]

In [ ]:
# ============================================================
# Metrics
# ============================================================
def safe_div(a,b):
    return a / b if b else 0.0

def metrics(rows):
    y_true = [r.get('gold') for r in rows]
    y_pred = [r.get('pred') for r in rows]
    n = len(rows)
    acc = safe_div(sum(yt == yp for yt, yp in zip(y_true, y_pred)), n)
    per = {}
    f1s = []
    weights = []
    for lab in LABELS:
        tp = sum(yt == lab and yp == lab for yt, yp in zip(y_true, y_pred))
        fp = sum(yt != lab and yp == lab for yt, yp in zip(y_true, y_pred))
        fn = sum(yt == lab and yp != lab for yt, yp in zip(y_true, y_pred))
        support = sum(yt == lab for yt in y_true)
        pred_count = sum(yp == lab for yp in y_pred)
        prec = safe_div(tp, tp + fp)
        rec = safe_div(tp, tp + fn)
        f1 = safe_div(2 * prec * rec, prec + rec)
        per[lab] = dict(precision=prec, recall=rec, f1=f1, support=support, pred_count=pred_count)
        f1s.append(f1)
        weights.append(support)
    macro = sum(f1s) / len(f1s)
    weighted = safe_div(sum(f*w for f,w in zip(f1s, weights)), sum(weights))
    mc_f1s = [per[x]['f1'] for x in ['A','B','C','D']]
    ynu_f1s = [per[x]['f1'] for x in ['Yes','No','Unknown']]
    return {
        'n': n,
        'acc': acc,
        'macro_f1': macro,
        'weighted_f1': weighted,
        'mc_macro': sum(mc_f1s)/4,
        'ynu_macro': sum(ynu_f1s)/3,
        'per_label': per,
        'gold_count': dict(Counter(y_true)),
        'pred_count': dict(Counter(y_pred)),
    }

def diffs(rows_a, rows_b):
    by_a = {int(r['idx']): r for r in rows_a}
    out = []
    for r in rows_b:
        i = int(r['idx'])
        if by_a[i].get('pred') != r.get('pred'):
            out.append(i)
    return sorted(out)

def row_by_idx(rows, idx):
    for r in rows:
        if int(r.get('idx')) == int(idx):
            return r
    raise KeyError(idx)

In [ ]:
# ============================================================
# Load baseline artifacts
# ============================================================
# Prefer v32.2 independent/full if present, then Claude v32.2 full.
v32_rows, v32_path = load_json([
    'v32_2_full_preds.json',
    'v32_2_independent_full_preds.json',
    'v32_2_independent_standard_preds.json',
], required=True)

v27_rows, v27_path = load_json('v27_standard_preds.json', required=False)

assert isinstance(v32_rows, list) and len(v32_rows) > 0
base_m = metrics(v32_rows)
print('Loaded baseline:', v32_path)
print('Baseline metrics:', {k: base_m[k] for k in ['acc','macro_f1','weighted_f1','mc_macro','ynu_macro']})

# Sanity: baseline should be v32.2. Tolerate tiny float differences, but fail if clearly wrong.
assert abs(base_m['macro_f1'] - BASELINE_MACRO_EXPECTED) < 1e-9, 'Baseline is not v32.2 macro-F1. STOP.'
assert abs(base_m['acc'] - BASELINE_ACC_EXPECTED) < 1e-9, 'Baseline is not v32.2 acc. STOP.'
if v27_rows:
    print('Loaded v27:', v27_path)
    print('baseline diffs vs v27:', diffs(v27_rows, v32_rows))

## Optional comparison to Claude v32.2 notebook

This cell does **not** depend on Claude's advocate artifacts. It only scans the hotfix notebook if available to label it as a surgical one-case repair vs the broader v32 idea.

In [ ]:
# ============================================================
# Optional: inspect Claude notebook structure
# ============================================================
claude_nb_info = {'found': False}
try:
    nb_path = find_file('notebook_v32_2_s3_hotfix_idx14.ipynb', required=False)
    if nb_path:
        with open(nb_path, 'r', encoding='utf-8') as f:
            nb = json.load(f)
        src = '\n'.join(''.join(c.get('source','')) for c in nb.get('cells', []))
        claude_nb_info = {
            'found': True,
            'path': str(nb_path),
            'hardcoded_idx14': '14' in src and 'idx' in src,
            'mentions_advocate_candidates': 'advocate' in src.lower(),
            'reloads_saved_preds': ('reload' in src.lower() or 'load_json' in src.lower()) and 'preds' in src.lower(),
            'mentions_relaxed_gate': 'relax' in src.lower() or 'relaxed' in src.lower(),
        }
except Exception as e:
    claude_nb_info = {'found': False, 'error': repr(e)}
print('Claude notebook scan:', claude_nb_info)

## v33 audit logic

The audit uses conservative, proof-oriented textual/Horn signals from the existing selected explanation and, if available, data context. It does **not** use gold labels in the selection rule.

This first version is intentionally strict and may rollback. That is acceptable: the goal is to avoid turning v32.2's single-case safety into broad validation overfit.

In [ ]:
# ============================================================
# Proof/Horn-style signal extraction
# ============================================================
FINAL_RE = re.compile(r'Final Answer\s*[:\-]?\s*(Yes|No|Unknown|[ABCD]|UNPARSEABLE)\b', re.I)
SUPPORT_RE = re.compile(r'Supporting Premises\s*:\s*\[([^\]]*)\]', re.I)
PREMISE_REF_RE = re.compile(r'\bpremise\s+(\d+)\b', re.I)

NEGATIVE_CUES = [
    'false', 'not true', 'cannot', "can't", 'does not', 'do not', 'did not', 'will not',
    'fails to', 'lack', 'lacks', 'insufficient', 'cannot determine', 'unknown', 'uncertain',
    'not enough', 'no evidence', 'not mentioned', 'not directly', 'doesn\'t follow', 'does not follow'
]
AFFIRMATIVE_CUES = [
    'therefore', 'thus', 'hence', 'so the statement is true', 'the statement is true',
    'is true', 'can ', 'qualifies', 'meets all requirements', 'will ', 'should ',
    'does follow', 'follows that', 'there exists', 'all ', 'every ', 'must ', 'is confirmed'
]
HORN_CUES = [
    'if ', 'then ', 'implies', 'leads to', 'from premise', 'therefore', 'thus', 'all ', 'every ', 'for all', 'forall', '∃', 'exists'
]


def strip_final_answer(text):
    if not text:
        return ''
    return re.split(r'Final Answer\s*[:\-]?', str(text), flags=re.I)[0]

def parse_final(text):
    vals = FINAL_RE.findall(str(text or ''))
    if not vals:
        return None
    v = vals[-1]
    return v.title() if v.lower() in ['yes','no','unknown'] else v.upper()

def support_premises(text):
    text = str(text or '')
    m = SUPPORT_RE.search(text)
    nums = set()
    if m:
        for x in re.findall(r'\d+', m.group(1)):
            nums.add(int(x))
    for x in PREMISE_REF_RE.findall(text):
        nums.add(int(x))
    return sorted(nums)

def last_reasoning_sentence(text):
    body = strip_final_answer(text)
    # Get last non-empty sentence-ish chunk.
    parts = re.split(r'(?<=[.!?])\s+', body.strip())
    parts = [p.strip() for p in parts if p.strip()]
    return parts[-1] if parts else body.strip()

def cue_count(text, cues):
    low = str(text or '').lower()
    return sum(1 for c in cues if c in low)

def has_negative_near_conclusion(text):
    s = last_reasoning_sentence(text).lower()
    return any(c in s for c in NEGATIVE_CUES)

def is_affirmative_reasoning(row):
    exp = str(row.get('explanation') or '')
    body = strip_final_answer(exp)
    last = last_reasoning_sentence(exp).lower()
    # Do not trust old final answer if it says No. We use the body before Final Answer.
    support = support_premises(exp)
    aff = cue_count(body, AFFIRMATIVE_CUES)
    horn = cue_count(body, HORN_CUES)
    neg_last = has_negative_near_conclusion(exp)
    # Strong signals for the known pattern: body proves true, final answer says No.
    true_phrase = any(p in body.lower() for p in [
        'the statement is true', 'making the statement true', 'so the statement is true',
        'therefore, the statement is true', 'thus, the statement is true', 'is true because',
        'therefore, all', 'therefore, every', 'therefore, if', 'therefore, there exists',
        'thus, all', 'thus, every', 'thus, if', 'thus, there exists'
    ])
    return {
        'supporting_premises': support,
        'support_count': len(support),
        'affirmative_cue_count': aff,
        'horn_cue_count': horn,
        'negative_near_conclusion': neg_last,
        'body_true_phrase': true_phrase,
        'old_inner_final': parse_final(exp),
        'last_reasoning_sentence': last_reasoning_sentence(exp)[:500],
    }

def should_flip_no_to_yes_v33(row):
    """Gold-free, conservative rule. Returns (bool, reason, details)."""
    idx = int(row['idx'])
    if idx in BANKED_INDICES:
        return False, 'banked_index_do_not_touch', {}
    if row.get('is_mc'):
        return False, 'mc_row_do_not_touch', {}
    if row.get('pred') != 'No':
        return False, 'not_current_no', {}
    # only YNU-style questions
    q = str(row.get('question') or '').lower()
    yq = any(q.startswith(x) for x in ['does ', 'do ', 'can ', 'is ', 'are ', 'should ', 'will ']) or 'statement:' in q
    if not yq:
        return False, 'not_yes_no_question_shape', {}
    sig = is_affirmative_reasoning(row)
    # Very conservative: require explicit premise refs + true phrase + horn cues, and no negative conclusion.
    if sig['support_count'] >= 1 and sig['body_true_phrase'] and sig['horn_cue_count'] >= 2 and not sig['negative_near_conclusion']:
        return True, 'R1_affirmative_horn_body_true_with_premises', sig
    # Slightly broader but still proof-like: many horn/affirm cues and positive final sentence.
    if sig['support_count'] >= 2 and sig['affirmative_cue_count'] >= 2 and sig['horn_cue_count'] >= 3 and not sig['negative_near_conclusion']:
        return True, 'R1b_affirmative_multi_premise_chain', sig
    return False, 'no_strict_proof_signal', sig

In [ ]:
# ============================================================
# v33_a audit
# ============================================================
audit_cases = []
for r in v32_rows:
    i = int(r['idx'])
    if r.get('is_mc') or i in BANKED_INDICES:
        continue
    if r.get('pred') == 'No':
        ok, reason, details = should_flip_no_to_yes_v33(r)
        audit_cases.append({
            'idx': i,
            'gold_for_audit_only': r.get('gold'),  # not used by rule; only for posthoc audit
            'pred': r.get('pred'),
            'question': str(r.get('question',''))[:300],
            'rule_would_flip': ok,
            'reason': reason,
            'details': details,
        })

would_flip = [c for c in audit_cases if c['rule_would_flip']]
by_gold = Counter(c['gold_for_audit_only'] for c in would_flip)
audit_summary = {
    'version': 'v33_a_horn_proof_audit',
    'baseline': 'v32.2',
    'baseline_macro_f1': base_m['macro_f1'],
    'banked_indices': sorted(BANKED_INDICES),
    'n_current_no_nonmc_examined': len(audit_cases),
    'n_rule_would_flip_no_to_yes': len(would_flip),
    'would_flip_indices': [c['idx'] for c in would_flip],
    'posthoc_gold_distribution_for_would_flip': dict(by_gold),
    'note': 'Gold is included only for audit; selection rule uses no gold labels.',
}
print(json.dumps(audit_summary, indent=2, ensure_ascii=False))
save_json(audit_summary, '/kaggle/working/v33_a_horn_audit_summary.json')
save_json(audit_cases, '/kaggle/working/v33_a_horn_audit_cases.json')

In [ ]:
# ============================================================
# v33_standard: apply conservative proof flips, then gate
# ============================================================
v33_candidate = copy_rows(v32_rows)
flip_log = []
for row in v33_candidate:
    ok, reason, details = should_flip_no_to_yes_v33(row)
    if ok:
        old = row.get('pred')
        row['pred'] = 'Yes'
        row['v33_old_pred'] = old
        row['v33_repair_rule'] = reason
        row['v33_repair_details'] = details
        flip_log.append({'idx': int(row['idx']), 'old': old, 'new': 'Yes', 'rule': reason, 'gold_for_audit_only': row.get('gold')})

cand_m = metrics(v33_candidate)
base_per = base_m['per_label']
cand_per = cand_m['per_label']
macro_gain = cand_m['macro_f1'] - base_m['macro_f1']
no_drop = base_per['No']['f1'] - cand_per['No']['f1']
unk_drop = base_per['Unknown']['f1'] - cand_per['Unknown']['f1']
mc_changed = abs(cand_m['mc_macro'] - base_m['mc_macro']) > 1e-12
collapse = any(cand_m['per_label'][lab]['pred_count'] == 0 and base_m['per_label'][lab]['support'] > 0 for lab in ['Yes','No','Unknown'])

selected = (
    len(flip_log) > 0 and
    len(flip_log) <= MAX_STANDARD_FLIPS and
    macro_gain > STRICT_MACRO_GAIN and
    no_drop <= MAX_NO_F1_DROP and
    unk_drop <= MAX_UNKNOWN_F1_DROP and
    not collapse and
    (not mc_changed or ALLOW_MC_CHANGE)
)

standard_rows = v33_candidate if selected else copy_rows(v32_rows)
standard_m = metrics(standard_rows)
standard_summary = {
    'version': 'v33_standard_horn_proof_rescue',
    'selection_status': 'SELECT_V33_STANDARD' if selected else 'ROLLBACK_v32_2',
    'selected_source': 'v33_horn_proof_candidate' if selected else 'ROLLBACK_v32_2',
    'baseline_v32_2': {k: base_m[k] for k in ['n','acc','macro_f1','weighted_f1','mc_macro','ynu_macro']},
    'candidate_metrics': {k: cand_m[k] for k in ['n','acc','macro_f1','weighted_f1','mc_macro','ynu_macro']},
    'metrics': {k: standard_m[k] for k in ['n','acc','macro_f1','weighted_f1','mc_macro','ynu_macro']},
    'per_label': standard_m['per_label'],
    'candidate_per_label': cand_m['per_label'],
    'n_candidate_flips': len(flip_log),
    'candidate_flips': flip_log,
    'selected_flips': flip_log if selected else [],
    'gates': {
        'macro_gain_candidate': macro_gain,
        'strict_macro_gain_required': STRICT_MACRO_GAIN,
        'no_drop': no_drop,
        'max_no_drop': MAX_NO_F1_DROP,
        'unknown_drop': unk_drop,
        'max_unknown_drop': MAX_UNKNOWN_F1_DROP,
        'mc_changed': mc_changed,
        'collapse': collapse,
        'n_flips_ok': len(flip_log) <= MAX_STANDARD_FLIPS,
        'selected': selected,
    },
    'note': 'If rollback, saved preds are exactly v32.2 baseline.',
}

save_json(standard_rows, '/kaggle/working/v33_standard_preds.json')
save_json(standard_summary, '/kaggle/working/v33_standard_summary.json')
print(json.dumps({k: standard_summary[k] for k in ['selection_status','metrics','n_candidate_flips','gates']}, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# v33_full: reload saved preds and verify select/rollback
# ============================================================
reloaded_rows, reload_path = load_json('v33_standard_preds.json', required=True)
reloaded_m = metrics(reloaded_rows)
actual_diffs_from_base = diffs(v32_rows, reloaded_rows)

# Verify summary metrics match saved preds.
assert abs(reloaded_m['macro_f1'] - standard_summary['metrics']['macro_f1']) < 1e-12, 'Saved preds macro != summary macro'
assert abs(reloaded_m['mc_macro'] - standard_summary['metrics']['mc_macro']) < 1e-12, 'Saved preds MC != summary MC'

if standard_summary['selection_status'] == 'SELECT_V33_STANDARD':
    expected = sorted(x['idx'] for x in standard_summary['selected_flips'])
    verified = (
        actual_diffs_from_base == expected and
        reloaded_m['macro_f1'] > base_m['macro_f1'] + STRICT_MACRO_GAIN and
        abs(reloaded_m['mc_macro'] - base_m['mc_macro']) < 1e-12 and
        reloaded_m['per_label']['Unknown']['f1'] >= base_m['per_label']['Unknown']['f1'] - 1e-12
    )
else:
    expected = []
    verified = (actual_diffs_from_base == [])

if verified and standard_summary['selection_status'] == 'SELECT_V33_STANDARD':
    full_rows = reloaded_rows
    full_status = 'SELECT_V33'
    source = 'v33_standard_horn_proof_rescue_verified'
else:
    full_rows = copy_rows(v32_rows)
    full_status = 'ROLLBACK_TO_V32_2'
    source = 'ROLLBACK_v32_2'

full_m = metrics(full_rows)
full_diffs = diffs(v32_rows, full_rows)
full_summary = {
    'version': 'v33_full_select_or_rollback',
    'selection_status': full_status,
    'source': source,
    'reason': f'verified={verified}; standard_status={standard_summary["selection_status"]}; actual_diffs={actual_diffs_from_base}; expected={expected}',
    'baseline_v32_2': {k: base_m[k] for k in ['n','acc','macro_f1','weighted_f1','mc_macro','ynu_macro']},
    'metrics': {k: full_m[k] for k in ['n','acc','macro_f1','weighted_f1','mc_macro','ynu_macro']},
    'per_label': full_m['per_label'],
    'n_flips_from_v32_2': len(full_diffs),
    'flipped_indices_from_v32_2': full_diffs,
    'diffs_vs_v27': diffs(v27_rows, full_rows) if v27_rows else None,
    'standard_summary_path': '/kaggle/working/v33_standard_summary.json',
}

save_json(full_rows, '/kaggle/working/v33_full_preds.json')
save_json(full_summary, '/kaggle/working/v33_full_summary.json')
print(json.dumps(full_summary, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# Risk report
# ============================================================
final_rows, _ = load_json('v33_full_preds.json')
final_summary, _ = load_json('v33_full_summary.json')
final_m = metrics(final_rows)

overfit = 'LOW_PROOF_GROUNDED_ROLLBACK' if final_summary['selection_status'].startswith('ROLLBACK') else 'MEDIUM_PROOF_RULE_VALIDATION_TUNED'
underfit = 'STILL_HIGH_YES_NO_REMAINING'
if final_summary['selection_status'] == 'SELECT_V33':
    underfit = 'MEDIUM_HIGH_REDUCED_BY_HORN_RESCUE'

risk_report = {
    'version': 'v33_unified_risk_report',
    'final_decision': final_summary['selection_status'],
    'metrics': {k: final_m[k] for k in ['n','acc','macro_f1','weighted_f1','mc_macro','ynu_macro']},
    'delta_vs_v32_2': {
        'acc': final_m['acc'] - base_m['acc'],
        'macro_f1': final_m['macro_f1'] - base_m['macro_f1'],
        'weighted_f1': final_m['weighted_f1'] - base_m['weighted_f1'],
        'mc_macro': final_m['mc_macro'] - base_m['mc_macro'],
        'ynu_macro': final_m['ynu_macro'] - base_m['ynu_macro'],
    },
    'n_flips_from_v32_2': final_summary['n_flips_from_v32_2'],
    'flipped_indices_from_v32_2': final_summary['flipped_indices_from_v32_2'],
    'overfit_risk': overfit,
    'underfit_risk': underfit,
    'artifact_risk': 'LOW_RELOADED_SAVED_PREDS',
    'label_collapse': any(final_m['per_label'][lab]['pred_count'] == 0 and final_m['per_label'][lab]['support'] > 0 for lab in ['Yes','No','Unknown']),
    'audit_summary': audit_summary,
    'recommendation': 'USE_V33' if final_summary['selection_status'] == 'SELECT_V33' else 'KEEP_V32_2',
}
save_json(risk_report, '/kaggle/working/v33_risk_report.json')
print(json.dumps(risk_report, indent=2, ensure_ascii=False))
print('\nDONE v33 unified. Final:', risk_report['recommendation'], 'macro=', final_m['macro_f1'])